# Module 7 Assignment: Data Science Ethics
**CS 5045: Computation for the Data Sciences**

**Name:** Brian Angelo Thomas

---

This assignment is structured differently from other modules. You will run a fairness audit
on a loan application dataset and reflect on what you find. Sections 1 and 2 give you most
of the code; fill in the blanks, run the cells, and interpret the results. Section 4 gives you
no code at all, you write the filter-group-summarize pipeline yourself from the question down.

Run all cells top to bottom and make sure there are no errors. Commit and push this notebook
(`hw7.ipynb`) to your fork on GitHub, no Canvas upload needed.

**Dataset:** `data.csv`, 18 loan applicants, 9 columns.
The file is in the `Datasets/` folder one level up from this notebook.

In [20]:
import pandas as pd

# Load the dataset; all sections use this DataFrame.
df = pd.read_csv('data.csv')

## Section 1: Read and Observe

Display the dataset, then fill in the two blanks to count applicants and approved loans.

In [21]:
# Display the full dataset.
print(df)

# 1a. How many applicants are in the dataset?
#     Use len() on the DataFrame.
n_applicants = len(df)
print(f"\nNumber of applicants: {n_applicants}")
# Expected: 18

# 1b. How many applicants were approved?
#     The 'approved' column contains 1 for approved, 0 for denied.
#     Use sum() on that column to count the 1s.
n_approved = df['approved'].sum()
print(f"Number approved: {n_approved}")
# Expected: 10

   applicant_id  age  income  zip_code race_category  credit_score  \
0          A001   34   52000     24060         White           720   
1          A002   27   38000     24061         Black           680   
2          A003   45   75000     22903         White           760   
3          A004   31   41000     24061      Hispanic           650   
4          A005   52   91000     22901         White           800   
5          A006   29   35000     24060         Black           670   
6          A007   41   63000     22902         Asian           740   
7          A008   38   57000     24061      Hispanic           700   
8          A009   24   29000     24060         Black           620   
9          A010   55  110000     22903         White           820   
10         A011   33   48000     22901      Hispanic           690   
11         A012   47   82000     22902         White           775   
12         A013   26   32000     24061         Black           640   
13         A014   36

## Section 2: The Fairness Audit

A fairness audit disaggregates an outcome by demographic group. Here we compute the
approval rate for each racial group and convert to a percentage.

Fill in the two blanks:
- The first blank is a number to multiply by (to convert a proportion to a percentage).
- The second blank is a number to round to (1 decimal place).

In [22]:
# Compute approval rate by racial group.
# .mean() gives a proportion (0 to 1); multiply to convert to a percentage.
approval_by_race = (
    df.groupby('race_category')['approved'].mean()
    *100
)
# Round to 1 decimal place.
approval_by_race = approval_by_race.round(1)

print('Approval rate by racial group (%):')
print(approval_by_race.to_string())

Approval rate by racial group (%):
race_category
Asian       100.0
Black         0.0
Hispanic     25.0
White       100.0


**Fill in the table below with the values from the output above.**

| Racial group | Approval rate (%) |
|---|-------------------|
| Asian | 100.0%            |
| Black | 0.0%              |
| Hispanic | 25.0%             |
| White | 100.00            |

## Section 3: Identify the Proxy

A proxy variable is a feature that correlates with a protected characteristic like race.
The crosstab below shows how many applicants of each racial group live in each zip code.

In [23]:
# Crosstab: zip code vs. race category.
# Look at which zip codes concentrate which racial groups.
print(pd.crosstab(df['zip_code'], df['race_category']))

race_category  Asian  Black  Hispanic  White
zip_code                                    
22901              0      0         1      2
22902              1      0         0      1
22903              1      0         0      2
24060              0      2         1      2
24061              0      3         2      0


**Write your interpretation below (2-3 sentences).**

What does the crosstab show about the relationship between zip code and race in this dataset?
If a model used zip code as a feature but removed race_category, what concern does that raise?

*Your answer:
The crosstab displays the count of individuals belonging to each racial group within specific ZIP codes. It reveals that racial groups are not distributed evenly across these locations. Instead, certain ZIP codes contain distinct concentrations of specific racial groups:
• ZIP code 24061 has exclusively Black and Hispanic individuals (5 total), with no Asian or White residents listed.
• ZIP codes 22901 and 22903 are predominantly White, with zero Black residents recorded.
This shows a strong correlation between geographic location (ZIP code) and race within the dataset.

## Section 4: The Missing Data Problem

So far you have audited *who* gets approved. A complete fairness audit also checks whether
the model is equally *reliable* for the groups it does approve, using an error-rate metric
like default rate. If approved borrowers from one group default far more (or less) than
approved borrowers from another, the model is not performing the same job across groups
even when the approval rates look fine.

There are no blanks in this section. Write the code yourself: filter, group, and summarize,
using the same tools Sections 1 and 2 already showed you.

In [24]:
# Write your own code below. No blanks to fill in this time.
#
# 1. Filter the dataset to approved applicants only (the 'approved' column equals 1).
# 2. Group the filtered data by 'race_category'.
# 3. Compute the mean 'default_rate' for each group, rounded to 3 decimal places.
# 4. Print the result.


In [25]:

# 1. Filter the dataset to approved applicants only (the 'approved' column equals 1).
approved_df = df[df['approved'] == 1]

# 2. Group the filtered data by 'race_category'.
# 3. Compute the mean 'default_rate' for each group, rounded to 3 decimal places.
mean_default_by_race = approved_df.groupby('race_category')['default_rate'].mean().round(3)

# 4. Print the result.
print("Mean default rate by racial group (approved applicants only):")
print(mean_default_by_race.to_string())


Mean default rate by racial group (approved applicants only):
race_category
Asian       0.03
Hispanic    0.05
White       0.02


**Write your interpretation below (2-3 sentences).**

Look closely at your output. One racial group has no row at all, not a zero, not a blank,
it is simply missing. Which group, and why did this happen? What does this tell you about
trying to audit a model's fairness *after* a group has already been filtered out at an
earlier stage?

*Your answer:
Black is not is list here.

*Because df[df['approved'] == 1] completely eliminates rows where approved == 0, any racial group that had zero approvals is entirely dropped from the resulting approved_df DataFrame. When you run .groupby('race_category'), pandas only identifies unique values that are actively present in the filtered data, causing that group's row to vanish entirely rather than showing a 0 or NaN.

When you audit a model's fairness after a group has already been filtered out, it exposes a critical flaw known as conditioning on a post-treatment variable or survival bias. It creates a "false fairness" -. It is only looking at the approved applicants not the entire population. Lastly, we need to evaluate the entire pipline; the whole process to make changes.

## Section 5: Written Reflection

Answer each question in 2-3 sentences. Write directly below each question in the space
provided. There is no single right answer; you are graded for genuine engagement with
the questions.

**5a.** The model was trained on historical loan decisions made by human loan officers.
Explain in plain English why this could create a feedback loop. What happens when the
model is retrained on data generated by its own past decisions?

*Your answer:*

Training this predictive model on historical decisions made by humans creates a feedback loop because the model does not learn who is actually good at paying back loans; instead, it learns how to copy the biases of those human loan officers.

**5b.** The approval rate differs substantially across racial groups in this dataset.
Does this difference, by itself, prove that the model is biased? What additional
information would you need before drawing a stronger conclusion?

*Your answer:*
No, a difference in approval rates by itself does not definitively prove the model is biased.
This is highlighting, the adverse impact of the model; not conclusive proof of an unfair algorithmic flaw.
More infromation would be needed to fix the model. I would try to see who has "credit worthiness" . Examples are Credit Scores (FICO), Debt-to-Income (DTI) ratios, and employment history. Try to eleimnate the proxy varibles like zip code

**5c.** A colleague suggests removing the `race_category` column from the dataset
before training the model, arguing that a model cannot be biased if it never sees race.
Based on what Section 3 showed, will this prevent disparate impact? Why or why not?

*Your answer:* By removing race, it will not prevent the disparate impact. The model still looks at other varibles like zip code. This is "fairness through blindness". Simply hiding a protected characteristic from a model does not prevent it from discriminating against certain groups.